In [7]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd

In [8]:
os.makedirs("dataset_outputs/A2AR/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="data/src/datasets/a2ar/data/A2AR.csv",
    store_dir="dataset_outputs/A2AR/data",
    name="A2ARDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [9]:
dataset.prepareDataset(
    split=RandomSplit(test_fraction=0.2, dataset=dataset),
    feature_calculators=[MorganFP(radius=2, nBits=4096)],
    recalculate_features=True,
)

In [10]:
from qsprpred.data.descriptors.sets import RDKitDescs

rdkit_descs = RDKitDescs()

dataset.addDescriptors([rdkit_descs])

dataset.descriptorSets

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = dataset.X
X1, X2, y1, y2 = train_test_split(dataset.X, dataset.y, test_size=0.25, random_state=42)
X3 = dataset.X_ind
y3 = dataset.y_ind

imp_mean = SimpleImputer(missing_values=pd.NA, strategy='mean')
X1 = imp_mean.fit_transform(X1)
X2 = imp_mean.transform(X2)
X3 = imp_mean.transform(X3)

scaler = StandardScaler()
X1 = scaler.fit_transform(X1)
X2 = scaler.transform(X2)
X3 = scaler.transform(X3)



In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X1, y1 = smote.fit_resample(X1, y1)
display(pd.DataFrame(X1))


,0,1,2,3,4,5,6,7,8,9,...,4303,4304,4305,4306,4307,4308,4309,4310,4311,4312
0,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,-0.603779,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,1.980336
1,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,-0.603779,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,2.943564,-0.181096,-0.225319,-0.258817
2,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,-0.603779,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.213839
3,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,-0.603779,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,1.331313
4,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,-0.603779,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,4.222915,-2.480443
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3533,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,1.656237,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.403167
3534,-0.040456,0.0,-0.099504,-0.020215,-0.070186,10.614778,-0.067184,-0.603779,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,2.308674,1.306710
3535,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,1.656237,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.168575
3536,-0.040456,0.0,-0.099504,-0.020215,-0.070186,-0.053551,-0.067184,-0.603779,-0.111386,0.0,...,-0.145865,-0.060746,-0.088443,-0.116896,-0.359386,0.0,-0.281971,-0.181096,-0.225319,0.589209


In [54]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
import numpy as np

# spojení X a y dohromady
X_combined = np.concatenate((X1, X2))
y_combined = np.concatenate((y1, y2))

# definuj split: -1 = train, 0 = validation
test_fold = np.concatenate((
    np.full(len(X1), -1),  # train indices
    np.zeros(len(X2))        # val indices
))

# vytvoř PredefinedSplit
ps = PredefinedSplit(test_fold)

model = RandomForestClassifier()
param_grid = {
    'n_estimators': [5, 10, 20, 30, 50],
    'max_depth': [None, 10, 20, 30, 40,],
    "class_weight": ["balanced", None],
    "criterion": ["gini", "entropy", "log_loss"],
    "max_features": ["sqrt", "log2", None],
    "random_state": [42]
}
print(len(param_grid))
grid = GridSearchCV(model, param_grid, cv=ps, scoring='matthews_corrcoef')
grid.fit(X_combined, y_combined)


6


/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed whe

GridSearchCV(cv=PredefinedSplit(test_fold=array([-1, -1, ...,  0,  0])),
             estimator=RandomForestClassifier(),
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [None, 10, 20, 30, 40],
                         'max_features': ['sqrt', 'log2', None],
                         'n_estimators': [5, 10, 20, 30, 50],
                         'random_state': [42]},
             scoring='matthews_corrcoef')

In [55]:
print("Best validation score (negative MCC):", grid.best_score_)
print("Best parameters:", grid.best_params_)


Best validation score (negative MCC): 0.599609394493094
Best parameters: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'n_estimators': 10, 'random_state': 42}


In [58]:
param_grid = {
    'n_estimators': [6,7,8,9,10,11,12,13,14],
    'max_depth': [ 10, 11, 12, 13, 14,15, 16, 17 ,18],
    "class_weight": ["balanced"],
    "criterion": ["entropy"],
    "max_features": ["sqrt"],
    "random_state": [42]
}
print(len(param_grid))
grid = GridSearchCV(model, param_grid, cv=ps, scoring='matthews_corrcoef')
grid.fit(X_combined, y_combined)

6


/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed whe

GridSearchCV(cv=PredefinedSplit(test_fold=array([-1, -1, ...,  0,  0])),
             estimator=RandomForestClassifier(),
             param_grid={'class_weight': ['balanced'], 'criterion': ['entropy'],
                         'max_depth': [10, 11, 12, 13, 14, 15, 16, 17, 18],
                         'max_features': ['sqrt'],
                         'n_estimators': [6, 7, 8, 9, 10, 11, 12, 13, 14],
                         'random_state': [42]},
             scoring='matthews_corrcoef')

In [59]:
print("Best validation score (negative MCC):", grid.best_score_)
print("Best parameters:", grid.best_params_)


Best validation score (negative MCC): 0.6010533279636241
Best parameters: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 15, 'max_features': 'sqrt', 'n_estimators': 10, 'random_state': 42}


In [60]:
clf = RandomForestClassifier(**grid.best_params_)
clf.fit(X1, y1)

/Users/krynekt/anaconda3/envs/ml/lib/python3.11/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=15, n_estimators=10, random_state=42)

In [61]:
from sklearn.metrics import matthews_corrcoef
print(matthews_corrcoef(y2, clf.predict(X2)))

0.6010533279636241


In [62]:
from sklearn.metrics import classification_report

report = classification_report(y2, clf.predict(X2), output_dict=True)  # nebo output_dict=False pro text
print(report)


{'False': {'precision': 1.0, 'recall': 0.37142857142857144, 'f1-score': 0.5416666666666666, 'support': 35.0}, 'True': {'precision': 0.972636815920398, 'recall': 1.0, 'f1-score': 0.9861286254728878, 'support': 782.0}, 'accuracy': 0.9730722154222766, 'macro avg': {'precision': 0.986318407960199, 'recall': 0.6857142857142857, 'f1-score': 0.7638976460697773, 'support': 817.0}, 'weighted avg': {'precision': 0.9738090453485326, 'recall': 0.9730722154222766, 'f1-score': 0.9670880274824132, 'support': 817.0}}
